# Corrective RAG powered by LangDB

#### Create tables and insert data

In [14]:
CREATE TABLE langdb.popqa
ENGINE = Memory AS
SELECT *
FROM url('https://raw.githubusercontent.com/AlexTMallen/adaptive-retrieval/main/data/popQA.tsv', TSV)

In [16]:
select * from langdb.popqa limit 5

,id,subj,prop,obj,subj_id,prop_id,obj_id,s_aliases,o_aliases,s_uri,o_uri,s_wiki_title,o_wiki_title,s_pop,o_pop,question,possible_answers
0,4222362,George Rankin,occupation,politician,1850297,22,2834605,"""[""""George James Rankin""""]""","""[""""political leader"""",""""political figure"""",""""polit."""",""""pol""""]""",http://www.wikidata.org/entity/Q5543720,http://www.wikidata.org/entity/Q82955,George Rankin,Politician,142,25692,What is George Rankin's occupation?,"""[""""politician"""", """"political leader"""", """"political figure"""", """"polit."""", """"pol""""]"""
1,4725190,John Mayne,occupation,journalist,2079053,22,663400,[],"""[""""journo"""",""""journalists""""]""",http://www.wikidata.org/entity/Q6247345,http://www.wikidata.org/entity/Q1930187,John Mayne,Journalist,236,24952,What is John Mayne's occupation?,"""[""""journalist"""", """"journo"""", """"journalists""""]"""
2,4382392,Henry Feilden,occupation,politician,1925450,22,2834605,"""[""""Henry Master Feilden""""]""","""[""""political leader"""",""""political figure"""",""""polit."""",""""pol""""]""",http://www.wikidata.org/entity/Q5725578,http://www.wikidata.org/entity/Q82955,Henry Feilden (Conservative politician),Politician,58,25692,What is Henry Feilden's occupation?,"""[""""politician"""", """"political leader"""", """"political figure"""", """"polit."""", """"pol""""]"""
3,4822110,Kathy Saltzman,occupation,politician,2122743,22,2834605,[],"""[""""political leader"""",""""political figure"""",""""polit."""",""""pol""""]""",http://www.wikidata.org/entity/Q6377295,http://www.wikidata.org/entity/Q82955,Kathy Saltzman,Politician,127,25692,What is Kathy Saltzman's occupation?,"""[""""politician"""", """"political leader"""", """"political figure"""", """"polit."""", """"pol""""]"""
4,4011112,Eleanor Davis,occupation,cartoonist,1752619,22,68412,"""[""""Eleanor McCutcheon Davis""""]""","""[""""graphic artist"""",""""animator"""",""""illustrator""""]""",http://www.wikidata.org/entity/Q5354261,http://www.wikidata.org/entity/Q1114448,Eleanor Davis,Cartoonist,317,9649,What is Eleanor Davis's occupation?,"""[""""cartoonist"""", """"graphic artist"""", """"animator"""", """"illustrator""""]"""


### Embeddings
Langdb supports convenient way to generate embeddings on the fly. You can use the built-in `embed()` function for development and testing. 

You can also use `OpenAI` or other providers.

In [29]:
CREATE TABLE embeddings_fallback_data (
  id `Int64`,
  embeddings `Array`(`Float32`),
  question `String`
) 
ENGINE = MergeTree
ORDER BY id;

In [ ]:
#### Spawn task
Spawn tasks to periodically calculate and insert embeddings for new questions into the fallback data embeddings table every 5 seconds

In [30]:
SPAWN TASK
    BEGIN
INSERT INTO embeddings_fallback_data
SELECT p.id, embed(question), p.question 
FROM langdb.popqa AS p 
LEFT JOIN embeddings_fallback_data AS pe ON p.id = pe.id
WHERE p.id != pe.id
ORDER BY p.id
LIMIT 10
END EVERY 5 second;

Spawned Task: 9302ba5c-16e5-4e30-8939-44818ac98484

In [15]:
CREATE TABLE langdb.pdf_contents (
    id `Int64`,
    content `String`,
)
ENGINE = MergeTree
ORDER BY content;

In [17]:
INSERT INTO langdb.pdf_contents (id, content)
SELECT 1 as id, arrayStringConcat(groupArray(content), '') as content from
    load_html('https://en.wikipedia.org/api/rest_v1/page/html/William_Shakespeare')

In [18]:
INSERT INTO langdb.pdf_contents (id, content)
SELECT 2 as id, arrayStringConcat(groupArray(content), '') as content from
    load_html('https://en.wikipedia.org/api/rest_v1/page/html/Agatha_Christie');


In [19]:
INSERT INTO langdb.pdf_contents (id, content)
SELECT 3 as id, arrayStringConcat(groupArray(content), '') as content from
    load_html('https://en.wikipedia.org/api/rest_v1/page/html/Barbara_Cartland');

In [20]:
INSERT INTO langdb.pdf_contents (id, content)
SELECT 4 as id, arrayStringConcat(groupArray(content), '') as content from
    load_html('https://en.wikipedia.org/api/rest_v1/page/html/Danielle_Steel');

In [21]:
INSERT INTO langdb.pdf_contents (id, content)
SELECT 5 as id, arrayStringConcat(groupArray(content), '') as content from
    load_html('https://en.wikipedia.org/api/rest_v1/page/html/Harold_Robbins');

In [31]:
CREATE PROMPT evaluator_qa(
  system "Given a question and possible answer, respond if it is relevant or not. 
Answer 'yes' if the score is below 0.1. If score is not provided, return 'no'.
Please use 'yes' or 'no' as the answer. 
question: {{question}}
possible answer: {{answer}}
similarity score: {{score}}"
);


Prompt already present

In [32]:
CREATE MODEL evaluator(
  provider 'OpenAI',
  model_name 'gpt-3.5-turbo',
  prompt_name "evaluator_qa",
  execution_options (retries 1),
  args ["question", "answer", "score"]
);

Model already present

In [22]:
CREATE PROMPT retrieval_evaluator_prompt (
  system "
Given a question, does the following document have exact
information to answer the question?
Question: {{question}}
Document: {{document}}
Think Step by step, and answer with yes or no only
  "
);

In [26]:
CREATE MODEL retrieval_evaluator (
  provider 'OpenAI',
  model_name 'gpt-3.5-turbo',
  prompt_name "retrieval_evaluator_prompt",
  execution_options (retries 1),
  args ["question", "document"]
);

In [ ]:
CREATE ENDPOINT retrieval_evaluator(question String, document String) AS
SELECT retrieval_evaluator($question, $document)
WITH description = "Endpoint for evaluating the relevancy of an question and document";

In [ ]:
CREATE ENDPOINT get_correct_documents(question String) AS 
 with tbl as (
    select id, content, retrieval_evaluator($question, content) answer from langdb.pdf_contents
)
select id, answer, content
from tbl where positionCaseInsensitive(answer, 'yes') > 0
with description = "Endpoint to get the relevant document to the question";

In [33]:
CREATE PROMPT fallback_prompt (
  system "Given a question, find the document relevant to the question and return possible answers and similarity scores. 
Question: {{input}}"
);



In [44]:
CREATE ENDPOINT endpoint_fallback_data(question String "Query to search question") AS
WITH tbl AS (
  SELECT CAST(embed($question) AS `Array`(`Float32`)) AS question_embed
)
SELECT cosineDistance(embeddings, question_embed) AS similarity, p.id, p.possible_answers
FROM embeddings_fallback_data AS pe
JOIN langdb.popqa AS p ON p.id = pe.id
CROSS JOIN tbl
ORDER BY similarity 
LIMIT 10
WITH description = "Returns top 10 results for the given query with similarity, id, and possible_answers";

In [34]:
CREATE MODEL IF NOT EXISTS fallback_model (
  provider 'OpenAI',
  model_name 'gpt-3.5-turbo',
  prompt_name "fallback_prompt",
  execution_options (retries 1),
  args ["input"],
  tools ["endpoint_fallback_data"]
);

In [36]:
CREATE PROMPT master_qa(
  system "Given a question, try to find a relevant answer based on relavent documents. 
Relavent documents can be fetched with get_correct_documents tool.
If you cannot get any document, try to use the fallback data source.
Fallback data source can only be used if the main data source doesn't provide a relevant answer.
You must verify the answer with the evaluator. If the evaluator returns 'no', do not provide an answer to the user.
question: {{question}}"
);

In [39]:
CREATE MODEL master(
  provider 'OpenAI',
  model_name 'gpt-3.5-turbo',
  prompt_name "master_qa",
  execution_options (retries 1),
  args ["question"],
  tools ["get_correct_documents", "endpoint_fallback_data", "evaluator"]
);

In [50]:
SELECT * from master('What is the religion of Patrick Sheridan?')

,msg_type,content
0,,
1,,
2,,
3,,
4,,
5,,
6,,
7,,
8,,
9,,


### Embeddings
Langdb supports convenient way to generate embeddings on the fly. You can use the built-in `embed()` function for development and testing. 

You can also use `OpenAI` or other providers.

In [9]:
CREATE TABLE embeddings_main_data (
  id `Int64`,
  embeddings `Array`(`Float32`),
  question `String`
) 
ENGINE = MergeTree
ORDER BY id;

In [10]:
CREATE TABLE embeddings_fallback_data (
  id `Int64`,
  embeddings `Array`(`Float32`),
  question `String`
) 
ENGINE = MergeTree
ORDER BY id;

Use `embed()` function to generate embeddings for each chunk and store them into `pdf_embeddings` table.

You can also use `SPAWN TASK` feature to run this in the background. 

In [ ]:
SPAWN TASK
    BEGIN
INSERT INTO embeddings_main_data
SELECT p.id, embed(question), p.question 
FROM langdb.main_data AS p 
LEFT JOIN embeddings_main_data AS pe ON p.id = pe.id
WHERE p.id != pe.id
ORDER BY p.id
LIMIT 10
END EVERY 5 second;

In [ ]:
SPAWN TASK
    BEGIN
INSERT INTO embeddings_fallback_data
SELECT p.id, embed(question), p.question 
FROM langdb.fallback_data AS p 
LEFT JOIN embeddings_fallback_data AS pe ON p.id = pe.id
WHERE p.id != pe.id
ORDER BY p.id
LIMIT 10
END EVERY 5 second;

#### Spawn task
Spawn tasks to periodically calculate and insert embeddings for new questions into the fallback data embeddings table every 5 seconds